In [1]:
import pennylane as qp
from pennylane import qcut
from pennylane import numpy as np
from cut import get_tape, grover_circuit, qft_circuit
from functools import partial

In [2]:
dev = qp.device("default.qubit", wires=30)

@qp.qnode(dev)
def circuit():
    qft_circuit(10)
    return qp.expval(qp.PauliZ(0))


In [3]:
print("QNode:", circuit)
print("QNode type:", type(circuit))

tape = circuit.construct([], {})
print("Tape:", tape)
print("Tape type:", type(tape))

QNode: <QNode: device='<default.qubit device (wires=30) at 0x723bc0a2e870>', interface='auto', diff_method='best'>
QNode type: <class 'pennylane.workflow.qnode.QNode'>
Tape: <QuantumScript: wires=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], params=0>
Tape type: <class 'pennylane.tape.qscript.QuantumScript'>


In [ ]:
tape_dag = qcut.tape_to_graph(tape)

print("DAG type:", type(tape_dag))
print("DAG nodes:", len(tape_dag.nodes))
print("DAG edges:", len(tape_dag.edges))

DAG type: <class 'networkx.classes.multidigraph.MultiDiGraph'>
DAG nodes: 61
DAG edges: 101


: 

In [ ]:
cut_edges = qcut.kahypar_cut(
    graph=tape_dag,
    num_fragments=5,
    fragment_weights=[10, 10, 10, 10, 10],
    seed=42,
)

print("Number of cut edges:", len(cut_edges))

for i, edge in enumerate(cut_edges):
    print(f"Cut {i}: {edge}")

In [ ]:
cut_graph = qcut.place_wire_cuts(
    graph=tape_dag,
    cut_edges=cut_edges,
)

print("Cut graph created")

Cut graph created


In [ ]:
wire_cuts = [
    node
    for node in cut_graph.nodes
    if isinstance(node.obj, qp.WireCut)
]

print("Number of WireCut nodes:", len(wire_cuts))

for i, node in enumerate(wire_cuts):
    print(f"WireCut {i}: wires={node.obj.wires}")

Number of WireCut nodes: 7
WireCut 0: wires=Wires([0])
WireCut 1: wires=Wires([0])
WireCut 2: wires=Wires([1])
WireCut 3: wires=Wires([2])
WireCut 4: wires=Wires([3])
WireCut 5: wires=Wires([8])
WireCut 6: wires=Wires([8])


In [ ]:
cut_tape = qcut.graph_to_tape(cut_graph)

print(cut_tape.draw())

 0: ──H──//─╭Rϕ─╭Rϕ─╭Rϕ──//─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ─╭Rϕ ···
 1: ────────╰●──│───│───────│───│───│───│───│───│───│───│───│───│───│───│───│───│───│───│───│── ···
 2: ────────────╰●──│───────│───│───│───│───│───│───│───│───│───│───│───│───│───│───│───│───│── ···
 3: ────────────────╰●──────│───│───│───│───│───│───│───│───│───│───│───│───│───│───│───│───│── ···
 4: ────────────────────────╰●──│───│───│───│───│───│───│───│───│───│───│───│───│───│───│───│── ···
 5: ────────────────────────────╰●──│───│───│───│───│───│───│───│───│───│───│───│───│───│───│── ···
 6: ────────────────────────────────╰●──│───│───│───│───│───│───│───│───│───│───│───│───│───│── ···
 7: ────────────────────────────────────╰●──│───│───│───│───│───│───│───│───│───│───│───│───│── ···
 8: ────────────────────────────────────────╰●──│───│───│───│───│───│───│───│───│───│───│───│── ···
 9: ────────────────────────────────────────────╰●──│───│───│───│───│───│───│───│───│───│───│── ···


In [ ]:
qcut.replace_wire_cut_nodes(cut_graph)
print("Wire cuts replaced.")

Wire cuts replaced.


In [ ]:
fragments, communication_graph = qcut.fragment_graph(
    cut_graph
)

print("Number of fragments:", len(fragments))

for i, fragment in enumerate(fragments):

    print("\n" + "=" * 60)
    print(f"FRAGMENT {i}")
    print("=" * 60)

    print("Wires:", list(fragment.nodes))
    print("Number of wires:", len(fragment.edges))

    fragment_tape = qcut.graph_to_tape(fragment)

    print("Tape wires:", fragment_tape.wires)
    print("Qubits:", len(fragment_tape.wires))
    print("Operations:", len(fragment_tape.operations))

    print(fragment_tape.draw())

Number of fragments: 4

FRAGMENT 0
Wires: [Wrapped(H(0)), Wrapped(MeasureNode(wires=[0]))]
Number of wires: 1
Tape wires: Wires([0])
Qubits: 1
Operations: 2
0: ──H──MeasureNode─┤  

FRAGMENT 1
Wires: [Wrapped(H(1)), Wrapped(H(2)), Wrapped(ControlledPhaseShift(1.5707963267948966, wires=Wires([2, 1]))), Wrapped(ControlledPhaseShift(0.39269908169872414, wires=Wires([3, 0]))), Wrapped(MeasureNode(wires=[1])), Wrapped(MeasureNode(wires=[2])), Wrapped(ControlledPhaseShift(1.5707963267948966, wires=Wires([1, 0]))), Wrapped(ControlledPhaseShift(0.7853981633974483, wires=Wires([2, 0]))), Wrapped(PrepareNode(wires=[0])), Wrapped(ControlledPhaseShift(0.7853981633974483, wires=Wires([3, 1]))), Wrapped(MeasureNode(wires=[0])), Wrapped(ControlledPhaseShift(1.5707963267948966, wires=Wires([3, 2]))), Wrapped(H(3)), Wrapped(MeasureNode(wires=[3]))]
Number of wires: 16
Tape wires: Wires([0, 1, 2, 3])
Qubits: 4
Operations: 14
0: ──PrepareNode─╭Rϕ─╭Rϕ─╭Rϕ──MeasureNode──────────────────────────────────────